In [1]:
import subprocess
import re
import pandas as pd

def get_scheduled_tasks():
    result = subprocess.run(
        ["schtasks", "/query", "/fo", "LIST", "/v"],
        capture_output=True,
        text=True,
        encoding="cp850"  # important pour Windows FR
    )
    return result.stdout.split("\n\n")


def extract_fixed_time_tasks(tasks):
    extracted = []

    for task in tasks:
        # Cherche une heure au format HH:MM
        time_match = re.search(r"\b([0-2]?\d:[0-5]\d)\b", task)
        
        # Ignore les tâches sans heure précise
        if not time_match:
            continue
        
        time_value = time_match.group(1)

        # Nom tâche
        name_match = re.search(r"Nom de la tâche:\s+(.*)", task)
        if not name_match:
            name_match = re.search(r"TaskName:\s+(.*)", task)

        # Action
        action_match = re.search(r"Tâche à exécuter:\s+(.*)", task)
        if not action_match:
            action_match = re.search(r"Task To Run:\s+(.*)", task)

        # Fréquence
        schedule_match = re.search(r"Planification:\s+(.*)", task)
        if not schedule_match:
            schedule_match = re.search(r"Schedule:\s+(.*)", task)

        if name_match:
            extracted.append({
                "Nom": name_match.group(1).strip(),
                "Heure": time_value,
                "Fréquence": schedule_match.group(1).strip() if schedule_match else "N/A",
                "Action": action_match.group(1).strip() if action_match else "N/A"
            })

    return extracted


# Récupération
tasks = get_scheduled_tasks()
fixed_time_tasks = extract_fixed_time_tasks(tasks)

# DataFrame propre
df = pd.DataFrame(fixed_time_tasks).sort_values("Heure")

df

,Nom,Heure,Fréquence,Action
145,\Microsoft\Windows\input\InputSettingsRestoreD...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
169,\Microsoft\Windows\Management\Autopilot\Detect...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
170,\Microsoft\Windows\Management\Autopilot\Detect...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
171,\Microsoft\Windows\Management\Autopilot\Remedi...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
172,\Microsoft\Windows\Management\Provisioning\Cel...,00:00,Les données de planification ne sont pas dispo...,%windir%\system32\ProvTool.exe /turn 7 /source...
...,...,...,...,...
238,\Microsoft\Windows\Sysmain\ResPriStaticDbSync,23:16,Les données de planification ne sont pas dispo...,Gestionnaire COM
90,\Microsoft\Windows\Data Integrity Scan\Data In...,23:18,Les données de planification ne sont pas dispo...,Gestionnaire COM
89,\Microsoft\Windows\Data Integrity Scan\Data In...,23:47,Les données de planification ne sont pas dispo...,Gestionnaire COM
30,\Microsoft\Office\Office Background Push Maint...,23:59,Les données de planification ne sont pas dispo...,C:\Program Files\Microsoft Office\root\vfs\Pro...


In [2]:
df[df["Heure"] == "00:00"]

,Nom,Heure,Fréquence,Action
145,\Microsoft\Windows\input\InputSettingsRestoreD...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
169,\Microsoft\Windows\Management\Autopilot\Detect...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
170,\Microsoft\Windows\Management\Autopilot\Detect...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
171,\Microsoft\Windows\Management\Autopilot\Remedi...,00:00,Les données de planification ne sont pas dispo...,Gestionnaire COM
172,\Microsoft\Windows\Management\Provisioning\Cel...,00:00,Les données de planification ne sont pas dispo...,%windir%\system32\ProvTool.exe /turn 7 /source...
...,...,...,...,...
54,\Microsoft\Windows\Application Experience\Mare...,00:00,Les données de planification ne sont pas dispo...,Actions multiples
55,\Microsoft\Windows\Application Experience\Micr...,00:00,Les données de planification ne sont pas dispo...,%windir%\system32\compattelrunner.exe -m:appra...
57,\Microsoft\Windows\Application Experience\Sdbi...,00:00,Les données de planification ne sont pas dispo...,%windir%\system32\sdbinst.exe -mm
58,\Microsoft\Windows\Application Experience\Sdbi...,00:00,Les données de planification ne sont pas dispo...,%windir%\system32\sdbinst.exe -mm


In [3]:
df.to_csv("scheduled_tasks.csv", index=False)

In [1]:
import psutil
import time
from datetime import datetime

print("Surveillance des nouveaux processus...")
print("Laisse tourner jusqu'après minuit.\n")

known_pids = set(p.pid for p in psutil.process_iter())

while True:
    current_pids = set(p.pid for p in psutil.process_iter())

    new_pids = current_pids - known_pids

    for pid in new_pids:
        try:
            proc = psutil.Process(pid)
            parent = proc.parent()

            print("\n------------------------------")
            print("Heure :", datetime.now())
            print("Nom :", proc.name())
            print("PID :", proc.pid)

            if parent:
                print("Parent :", parent.name())
                print("PID parent :", parent.pid)

            print("Chemin :", proc.exe())
        except:
            pass

    known_pids = current_pids
    time.sleep(1)

Surveillance des nouveaux processus...
Laisse tourner jusqu'après minuit.


------------------------------
Heure : 2026-02-26 23:16:40.026714
Nom : python.exe
PID : 30792
Parent : Code.exe
PID parent : 17044
Chemin : C:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\Scripts\python.exe

------------------------------
Heure : 2026-02-26 23:16:40.059820
Nom : conhost.exe
PID : 16168
Parent : python.exe
PID parent : 30792
Chemin : C:\Windows\System32\conhost.exe

------------------------------
Heure : 2026-02-26 23:16:40.090742
Nom : python.exe
PID : 31880
Parent : Code.exe
PID parent : 17044
Chemin : C:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\Scripts\python.exe

------------------------------
Heure : 2026-02-26 23:16:40.126490
Nom : conhost.exe
PID : 14188
Parent : python.exe
PID parent : 31880
Chemin : C:\Windows\System32\conhost.exe

------------------------------
Heure : 2026-02-26 23:16:40.162795
Nom : python3.9.exe
PID : 18924
Parent : python.exe
PID 

KeyboardInterrupt: 